In [1]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()                      # reads your .env file
token = os.getenv("GITHUB_TOKEN")

print("Token loaded:", token is not None)

resp = requests.get(
    "https://api.github.com/rate_limit",
    headers={"Authorization": f"Bearer {token}"},
)
print("Status:", resp.status_code)
print("Requests allowed per hour:", resp.json()["resources"]["core"]["limit"])

Token loaded: True
Status: 200
Requests allowed per hour: 5000


In [5]:
REPO = "pandas-dev/pandas"
headers = {"Authorization": f"Bearer {token}"}

resp = requests.get(
    f"https://api.github.com/repos/{REPO}/issues",
    headers=headers,
    params={"state": "closed", "per_page": 5},
)
issues = resp.json()

for issue in issues:
    print("Number:      ", issue["number"])
    print("Title:       ", issue["title"])
    print("Is a PR?:    ", "pull_request" in issue)
    print("Created:     ", issue["created_at"])
    print("Closed:      ", issue["closed_at"])
    print("Close reason:", issue.get("state_reason"))
    print("Comments:    ", issue["comments"])
    print("Labels:      ", [label["name"] for label in issue["labels"]])
    print("-" * 50)

Number:       71737
Title:        11929 (792×419)
Is a PR?:     False
Created:      2026-10-26T21:54:24Z
Closed:       2026-10-26T22:23:00Z
Close reason: completed
Comments:     0
Labels:       ['Needs Triage']
--------------------------------------------------
Number:       71736
Title:        ENH: pandas to_sql add support for UPSERT operations (INSERT ... ON DUPLICATE KEY UPDATE)
Is a PR?:     False
Created:      2026-10-26T20:45:11Z
Closed:       2026-10-26T21:04:19Z
Close reason: completed
Comments:     1
Labels:       ['Enhancement', 'Needs Triage']
--------------------------------------------------
Number:       71732
Title:        CI: Bump pypa/cibuildwheel to v4.3.4
Is a PR?:     True
Created:      2026-10-26T17:01:29Z
Closed:       2026-10-26T21:26:41Z
Close reason: None
Comments:     2
Labels:       ['Build', 'CI']
--------------------------------------------------
Number:       71727
Title:        Backport PR #71703: CI: Move package-checks.yml to wheel.yml and speed up the

In [ ]:
import time
import pandas as pd
from datetime import datetime, timedelta, timezone

CUTOFF_NEW = datetime.now(timezone.utc) - timedelta(days=180)   # must be at least 6 months old
CUTOFF_OLD = datetime(2024, 1, 1, tzinfo=timezone.utc)            # not older than Jan 2024

rows = []
page = 1

while True:
    resp = requests.get(
        f"https://api.github.com/repos/{REPO}/issues",
        headers=headers,
        params={"state": "all", "sort": "created", "direction": "desc",
                "per_page": 100, "page": page},
    )
        if not isinstance(batch, list):
        print("Stopped by GitHub:", batch.get("message"))
        break
    batch = resp.json()
    if not batch:
        break

    for issue in batch:
        if "pull_request" in issue:
            continue                                   # skip PRs
        created = datetime.fromisoformat(issue["created_at"].replace("Z", "+00:00"))
        if created > CUTOFF_NEW:
            continue                                   # too new, skip
        rows.append({
            "number": issue["number"],
            "title": issue["title"],
            "body": issue["body"] or "",
            "created_at": issue["created_at"],
            "closed_at": issue["closed_at"],
            "state": issue["state"],
            "state_reason": issue.get("state_reason"),
            "author": issue["user"]["login"],
            "author_association": issue["author_association"],
            "labels": [l["name"] for l in issue["labels"]],
            "comments": issue["comments"],
        })

    oldest = datetime.fromisoformat(batch[-1]["created_at"].replace("Z", "+00:00"))
    print(f"page {page}: {len(rows)} issues kept so far, oldest on page {oldest.date()}")
    if oldest < CUTOFF_OLD:
        break
    page += 1
    time.sleep(0.5)                                    # be polite to GitHub's servers

df = pd.DataFrame(rows)
df = df[pd.to_datetime(df["created_at"]) >= CUTOFF_OLD]   # trim anything older than Jan 2024
df.to_json("issues_raw.json", orient="records", lines=True)
print("Saved", len(df), "issues")

page 1: 0 issues kept so far, oldest on page 2026-10-22
page 2: 0 issues kept so far, oldest on page 2026-10-20
page 3: 0 issues kept so far, oldest on page 2026-10-15
page 4: 0 issues kept so far, oldest on page 2026-10-11
page 5: 0 issues kept so far, oldest on page 2026-10-09
page 6: 0 issues kept so far, oldest on page 2026-10-05
page 7: 0 issues kept so far, oldest on page 2026-10-02
page 8: 0 issues kept so far, oldest on page 2026-09-28
page 9: 0 issues kept so far, oldest on page 2026-09-25
page 10: 0 issues kept so far, oldest on page 2026-09-20
page 11: 0 issues kept so far, oldest on page 2026-09-13
page 12: 0 issues kept so far, oldest on page 2026-09-11
page 13: 0 issues kept so far, oldest on page 2026-09-05
page 14: 0 issues kept so far, oldest on page 2026-09-04
page 15: 0 issues kept so far, oldest on page 2026-08-30
page 16: 0 issues kept so far, oldest on page 2026-08-23
page 17: 0 issues kept so far, oldest on page 2026-08-15
page 18: 0 issues kept so far, oldest on

TypeError: string indices must be integers, not 'str'

In [7]:
df = pd.DataFrame(rows)
df.to_json("issues_raw.json", orient="records", lines=True)

print("Saved", len(df), "issues")
print("From", df["created_at"].min()[:10], "to", df["created_at"].max()[:10])
print(df["state_reason"].value_counts(dropna=False))

Saved 1920 issues
From 2024-05-01 to 2026-04-29
state_reason
completed      1557
NaN             296
not_planned      49
reopened         10
duplicate         8
Name: count, dtype: int64


In [8]:
df = pd.read_json("issues_raw.json", lines=True)

df["created_at"] = pd.to_datetime(df["created_at"])
df["closed_at"] = pd.to_datetime(df["closed_at"])

# How many days it took to close (empty if still open)
df["days_to_close"] = (df["closed_at"] - df["created_at"]).dt.days

# Rule 2: keep issues still open on day 7
#   → either it was never closed, OR it closed after day 7
still_open_day7 = df["closed_at"].isna() | (df["days_to_close"] > 7)
df = df[still_open_day7]

# Rule 4: label = 1 if fixed, 0 if not
#   → fixed means: currently closed AND close reason is "completed"
df["fixed"] = ((df["state"] == "closed") & (df["state_reason"] == "completed")).astype(int)

print("Issues left:", len(df))
print(df["fixed"].value_counts())

Issues left: 1212
fixed
1    874
0    338
Name: count, dtype: int64


In [10]:
from tqdm.auto import tqdm   # progress bar; if it errors, run: pip install tqdm

def first_week_comments(issue_number, created_at):
    resp = requests.get(
        f"https://api.github.com/repos/{REPO}/issues/{issue_number}/comments",
        headers=headers,
        params={"per_page": 100},
    )
    comments = resp.json()
    if not isinstance(comments, list):
        return []
    cutoff = created_at + pd.Timedelta(days=7)
    keep = []
    for c in comments:
        c_time = pd.to_datetime(c["created_at"])
        if c_time <= cutoff:                       # only the first 7 days
            keep.append({
                "author": c["user"]["login"],
                "author_association": c["author_association"],
                "created_at": c["created_at"],
                "body": c["body"] or "",
            })
    return keep

all_comments = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    all_comments.append(first_week_comments(row["number"], row["created_at"]))
    time.sleep(0.2)

df["week1_comments"] = all_comments
df.to_json("issues_week1.json", orient="records", lines=True, date_format="iso")
print("Saved", len(df), "issues with first-week comments")

  0%|          | 0/1212 [00:00<?, ?it/s]

Saved 1212 issues with first-week comments


In [11]:
df["n_week1_comments"] = df["week1_comments"].apply(len)

print("Issues with at least one first-week comment:", (df["n_week1_comments"] > 0).sum())
print()
print("Who comments? (author_association counts)")
assoc = [c["author_association"] for comments in df["week1_comments"] for c in comments]
print(pd.Series(assoc).value_counts())

Issues with at least one first-week comment: 988

Who comments? (author_association counts)
MEMBER         1580
CONTRIBUTOR     999
NONE            892
Name: count, dtype: int64


In [14]:
# Idea 1: how specific is the description?
df["body_length"]   = df["body"].str.len()                       # number of characters
df["has_code"]      = df["body"].str.contains("```").astype(int)  # has a code block
df["has_traceback"] = df["body"].str.contains("traceback").astype(int) # has an error message
df["mentions_version"] = df["body"].str.contains(r"\d+\.\d+\.\d+").astype(int)  # e.g. 2.2.3

# Idea 2: was it opened by a pandas team member?
df["author_is_member"] = (df["author_association"] == "MEMBER").astype(int)

print(df[["body_length", "has_code", "has_traceback", "mentions_version", "author_is_member"]].head())

   body_length  has_code  has_traceback  mentions_version  author_is_member
0         1428         1              0                 0                 1
1         1012         0              0                 1                 1
2         1861         0              0                 0                 1
3         1687         1              0                 0                 1
4          529         0              0                 0                 1


In [15]:
# Ideas 3 and 7: did a member reply in week 1, and how fast?
def member_reply_hours(row):
    member_times = [
        pd.to_datetime(c["created_at"])
        for c in row["week1_comments"]
        if c["author_association"] == "MEMBER"
    ]
    if not member_times:
        return 168                                   # no reply → slowest possible (your Option B!)
    first = min(member_times)
    return (first - row["created_at"]).total_seconds() / 3600

df["hours_to_member_reply"] = df.apply(member_reply_hours, axis=1)
df["member_replied_week1"]  = (df["hours_to_member_reply"] < 168).astype(int)
df["n_week1_comments"]      = df["week1_comments"].apply(len)

# Idea 6: title prefix like BUG:, ENH:, DOC:
df["title_type"] = df["title"].str.extract(r"^([A-Za-z]+):")[0].str.upper().fillna("NONE")
print(df["title_type"].value_counts().head(10))

title_type
BUG      644
ENH      222
DOC      113
NONE      89
PERF      32
API       29
DEPR      14
QST       13
BUILD     11
CI         7
Name: count, dtype: int64


In [16]:
# Group rare title types into OTHER
keep = ["BUG", "ENH", "DOC", "NONE"]
df["title_type"] = df["title_type"].where(df["title_type"].isin(keep), "OTHER")

# Fix rate for each title type (mean of 0/1 = % fixed)
print(df.groupby("title_type")["fixed"].agg(["count", "mean"]).round(2))
print()
print(df.groupby("member_replied_week1")["fixed"].agg(["count", "mean"]).round(2))
print()
print(df.groupby("author_is_member")["fixed"].agg(["count", "mean"]).round(2))

            count  mean
title_type             
BUG           644  0.74
DOC           113  0.94
ENH           222  0.56
NONE           89  0.70
OTHER         144  0.72

                      count  mean
member_replied_week1             
0                       480  0.73
1                       732  0.72

                  count  mean
author_is_member             
0                   977  0.74
1                   235  0.64


In [17]:
from sklearn.model_selection import train_test_split

feature_cols = [
    "body_length", "has_code", "has_traceback", "mentions_version",
    "author_is_member", "member_replied_week1", "hours_to_member_reply",
    "n_week1_comments",
]

# One-hot encode title_type (Day 1!)
X = pd.get_dummies(df[feature_cols + ["title_type"]], columns=["title_type"], dtype=int)
y = df["fixed"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)

print("Features:", list(X.columns))
print("Train size:", len(X_train), " Test size:", len(X_test))
print("Fixed rate in train:", round(y_train.mean(), 2), " in test:", round(y_test.mean(), 2))

Features: ['body_length', 'has_code', 'has_traceback', 'mentions_version', 'author_is_member', 'member_replied_week1', 'hours_to_member_reply', 'n_week1_comments', 'title_type_BUG', 'title_type_DOC', 'title_type_ENH', 'title_type_NONE', 'title_type_OTHER']
Train size: 969  Test size: 243
Fixed rate in train: 0.72  in test: 0.72


In [18]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, roc_auc_score

def evaluate(name, model):
    model.fit(X_train, y_train)
    prob = model.predict_proba(X_test)[:, 1]
    pred = model.predict(X_test)
    print(f"{name:22s} accuracy={accuracy_score(y_test, pred):.2f}  AUC={roc_auc_score(y_test, prob):.2f}")

evaluate("Lazy (always fixed)", DummyClassifier(strategy="most_frequent"))
evaluate("Logistic regression", make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)))

Lazy (always fixed)    accuracy=0.72  AUC=0.50
Logistic regression    accuracy=0.72  AUC=0.66


In [19]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

evaluate("Decision tree (depth 3)", DecisionTreeClassifier(max_depth=3, random_state=0))
evaluate("Random forest", RandomForestClassifier(n_estimators=300, max_depth=6, random_state=0))
evaluate("XGBoost", XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05))

XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Users/shaiksameerpasha/Documents/github-issue-predictor/.venv/lib/python3.14/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <09B43001-8527-34F6-AAE4-B97DA8A6CBC0> /Users/shaiksameerpasha/Documents/github-issue-predictor/.venv/lib/python3.14/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Vol

In [20]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

evaluate("Decision tree (depth 3)", DecisionTreeClassifier(max_depth=3, random_state=0))
evaluate("Random forest", RandomForestClassifier(n_estimators=300, max_depth=6, random_state=0))
evaluate("Boosting (sklearn)", HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0))

Decision tree (depth 3) accuracy=0.72  AUC=0.67
Random forest          accuracy=0.72  AUC=0.68
Boosting (sklearn)     accuracy=0.70  AUC=0.67


In [21]:
from sklearn.model_selection import cross_val_score

models = {
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Decision tree":       DecisionTreeClassifier(max_depth=3, random_state=0),
    "Random forest":       RandomForestClassifier(n_estimators=300, max_depth=6, random_state=0),
    "Boosting":            HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0),
}

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring="roc_auc")
    print(f"{name:22s} AUC per fold: {scores.round(2)}  mean={scores.mean():.2f}  spread={scores.std():.2f}")

Logistic regression    AUC per fold: [0.55 0.61 0.66 0.62 0.65]  mean=0.62  spread=0.04
Decision tree          AUC per fold: [0.57 0.59 0.58 0.61 0.58]  mean=0.59  spread=0.01
Random forest          AUC per fold: [0.55 0.58 0.65 0.6  0.61]  mean=0.60  spread=0.03
Boosting               AUC per fold: [0.5  0.61 0.65 0.58 0.55]  mean=0.58  spread=0.05
